<a href="https://colab.research.google.com/github/faiqhaziem27/-Malaysian-based-Personal-Information-Detection-and-Extractor-in-Text-with-Error-Analysis/blob/model-training/ner_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Check whether currently using GPU

In [2]:
import torch
print(torch.cuda.is_available())

True


### Clone and install code from GitHub, and install seqeval for inference

In [3]:
%cd /content
!rm -rf project
!git clone -b model-training https://github.com/faiqhaziem27/-Malaysian-based-Personal-Information-Detection-and-Extractor-in-Text-with-Error-Analysis.git project
%cd /content/project
!pip install -q seqeval
!ls

/content
Cloning into 'project'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 125 (delta 56), reused 102 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 29.57 KiB | 3.70 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/project
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
DATA_DOCUMENTATION.md  model		 src
DATA_SETUP_GUIDE.md    README.md	 TOKENIZATION_GUIDE.md
DATASET_USAGE.md       requirements.txt


In [1]:
# pull recent commits from GitHub - if pulled, don't run this
%cd /content/project
!git pull

[Errno 2] No such file or directory: '/content/project'
/content
fatal: not a git repository (or any of the parent directories): .git


### Data preprocessing - mounting from Google Drive, copying processed data

In [4]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p data/processed
!cp -r /content/drive/MyDrive/ner_data/processed/* data/processed/
!ls data/processed/conll2003_3class


# unmount from Google Drive
drive.flush_and_unmount()

Mounted at /content/drive
dataset_dict.json  test  train	validation


### Verifying installation works

In [ ]:
!python -c "from src.load_processed_dataset import load_ner_dataset; print(load_ner_dataset())"

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_tags_3class'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_tags_3class'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'ner_tags_3class'],
        num_rows: 3453
    })
})


In [ ]:
# Checking the existence of train.py
!pwd
!ls model

/content/project
create_model.py  __init__.py  model_config.py  model-documentation.md  train.py


### Model training

In [ ]:
!python -m model.train

config.json: 100% 466/466 [00:00<00:00, 2.86MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 208kB/s]
vocab.txt: 100% 996k/996k [00:00<00:00, 13.8MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 9.21MB/s]
Map: 100% 14041/14041 [00:01<00:00, 9944.63 examples/s]
Map: 100% 3250/3250 [00:00<00:00, 9398.60 examples/s]
Map: 100% 3453/3453 [00:00<00:00, 10857.06 examples/s]

model.safetensors: downloading bytes:  61% 330M/542M [00:01<00:01, 195MB/s, 29.3MB/s  ]
model.safetensors: downloading bytes:  91% 492M/542M [00:02<00:00, 231MB/s, 40.6MB/s  ]
model.safetensors: reconstructing file:  62% 335M/542M [00:02<00:01, 120MB/s, 25.3MB/s  ]
model.safetensors: downloading bytes: 100% 511M/511M [00:07<00:00, 64.8MB/s, 44.1MB/s  ]
model.safetensors: reconstructing file: 100% 542M/542M [00:07<00:00, 68.7MB/s, 36.5MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 6487.61it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key         

In [ ]:
import pandas as pd
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="models/checkpoints/conll_v1",
    aggregation_strategy="simple",
)

texts = [
    "Iman, Jack, and Faiq are totally gonna ace the Airost internship.",
    "Universiti Teknologi Malaysia (UTM) is one of the best technical universities in Malaysia",
]

row = []
for i, text in enumerate(texts):
    for ent in ner(text):
        row.append({
            "sentence": i,
            "entity": ent["word"],
            "type": ent["entity_group"],
            "score": round(float(ent["score"]), 3),
            "start": ent["start"],
            "end": ent["end"],
        })

df = pd.DataFrame(row)
print(df.to_string(index=False))

# show only uncertain predictions
# df[df["score"] < 0.9]

# count entities per type
# df["type"].value_counts()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

 sentence                        entity type  score  start  end
        0                           Ima  PER  0.984      0    3
        0                           ##n  PER  0.701      3    4
        0                          Jack  PER  0.993      6   10
        0                            Fa  PER  0.993     16   18
        0                          ##iq  PER  0.743     18   20
        0                        Airost  ORG  0.956     47   53
        1 Universiti Teknologi Malaysia  ORG  0.914      0   29
        1                           UTM  ORG  0.739     31   34
        1                      Malaysia  LOC  0.989     81   89


### Saving training results into **models/checkpoint**

In [ ]:
RUN_NAME = "conll_v2"   # change the names based on the runs made

!zip -r {RUN_NAME}.zip models/checkpoints/{RUN_NAME}

from google.colab import files
files.download(f"{RUN_NAME}.zip")

  adding: models/checkpoints/final/ (stored 0%)
  adding: models/checkpoints/final/model.safetensors (deflated 7%)
  adding: models/checkpoints/final/training_args.bin (deflated 53%)
  adding: models/checkpoints/final/tokenizer_config.json (deflated 44%)
  adding: models/checkpoints/final/config.json (deflated 52%)
  adding: models/checkpoints/final/tokenizer.json (deflated 67%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# changing folder name to adapt to changes (saving feature for multiple checkpoints) - don't run this cell
!mv models/checkpoints/final models/checkpoints/conll_v1
!zip -r conll_v1.zip models/checkpoints/conll_v1

mv: cannot stat 'models/checkpoints/final': No such file or directory
  adding: models/checkpoints/conll_v1/ (stored 0%)
  adding: models/checkpoints/conll_v1/model.safetensors (deflated 7%)
  adding: models/checkpoints/conll_v1/training_args.bin (deflated 53%)
  adding: models/checkpoints/conll_v1/tokenizer_config.json (deflated 44%)
  adding: models/checkpoints/conll_v1/config.json (deflated 52%)
  adding: models/checkpoints/conll_v1/tokenizer.json (deflated 67%)
